In [1]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

## Install libraries

```bash
uv sync --upgrade
```

## Update repository

In [2]:
# ! git pull

## Add import path

In [3]:
import gc
import os
import sys
from pathlib import Path

In [4]:
def add_library_level(level=4):
    suf_path = ['..']
    path = '..'
    for i in range(0, level):
        join_path = suf_path * i
        path = '/'.join(join_path)
        module_path = os.path.abspath(os.path.join(path))
        if module_path not in sys.path:
            sys.path.append(module_path)
            print(f'Appendeding {path}')

In [5]:
add_library_level(level=5)

Appendeding 
Appendeding ..
Appendeding ../..
Appendeding ../../..
Appendeding ../../../..


## Organize imports

In [6]:
import multiprocessing

In [7]:
from src.lattmc.utils import init_device, empty_cache
from src.lattmc.paths import tr_paths, sae_paths, save_tmp, load_tmp

In [8]:
from src.lattmc.fca.fca_utils import *
from src.lattmc.fca.info_utils import FCAtoPCA
from src.lattmc.fca.lattice_utils import (diff, subset, upper_indices,
                                          upper_mask)
from src.lattmc.fca.utils import *
from src.lattmc.latex.latex_utils import print_latex_text

In [9]:
from src.lattmc.tc.tokenization_utils import BG_CODES

In [10]:
from src.lattmc.tc.transcoder_analyzers import ConceptAnalysis, init_analyzer

In [11]:
from src.lattmc.sae.nlp_sae_utils import gen_concept

In [12]:
import logging

#### Number of CPU cores

In [13]:
workers = multiprocessing.cpu_count()
workers

14

In [14]:
SEED = 2026

In [15]:
logging.basicConfig(level=logging.INFO)

In [16]:
device = init_device()
device

INFO:src.lattmc.utils:Device: mps


device(type='mps')

In [17]:
torch.__version__, np.__version__

('2.11.0', '2.4.4')

## Initialize Paths

In [18]:
GPT2, TOKENS_PATH = tr_paths()

In [19]:
! ls {GPT2}

layer_error              v_min_nonzero0.npz       v_min4.npz
owt_tokens               v_min_nonzero1.npz       v_min5.npz
owt_tokens_canonical     v_min_nonzero10.npz      v_min6.npz
owt_tokens_with_texts    v_min_nonzero11.npz      v_min7.npz
qa_synthetic_general.csv v_min_nonzero2.npz       v_min8.npz
qa_synthetic.csv         v_min_nonzero3.npz       v_min9.npz
v_max0.npz               v_min_nonzero4.npz       V0.npz
v_max1.npz               v_min_nonzero5.npz       V1.npz
v_max10.npz              v_min_nonzero6.npz       V10.npz
v_max11.npz              v_min_nonzero7.npz       V11.npz
v_max2.npz               v_min_nonzero8.npz       V2.npz
v_max3.npz               v_min_nonzero9.npz       V3.npz
v_max4.npz               v_min0.npz               V4.npz
v_max5.npz               v_min1.npz               V5.npz
v_max6.npz               v_min10.npz              V6.npz
v_max7.npz               v_min11.npz              V7.npz
v_max8.npz               v_min2.npz               V8.npz
v_max

In [20]:
GPT2.absolute()

PosixPath('/Users/ltsinadze/git/my_papers/notebooks/transcoders/data/transcoders/gpt2')

In [21]:
! ls {TOKENS_PATH}

/Users/ltsinadze/git/my_papers/notebooks/transcoders/data/transcoders/gpt2/owt_tokens/owt_tokens_torch.pt


In [22]:
! ls {str(TOKENS_PATH.absolute())}

/Users/ltsinadze/git/my_papers/notebooks/transcoders/data/transcoders/gpt2/owt_tokens/owt_tokens_torch.pt


## Load trancoders

In [23]:
layers = list(range(12))
# layers = [0, 4, 6, 8, 10, 11]
# layers = [0, 5, 7, 9, 11]
layers = [1, 3, 9]

In [24]:
tr_analyzer = init_analyzer(
    layers,
    TOKENS_PATH,
    GPT2,
    model_name='gpt2-small',
    device=device,
    tr_or_sae=True,
    vector_dir=GPT2,
)

INFO:src.lattmc.tc.transcoder_utils:Loading transcoder for layer 1
INFO:httpx:HTTP Request: HEAD https://huggingface.co/pchlenski/gpt2-transcoders/resolve/main/final_sparse_autoencoder_gpt2-small_blocks.1.ln2.hook_normalized_24576.pt "HTTP/1.1 302 Found"
INFO:src.lattmc.tc.transcoder_utils:Loaded file final_sparse_autoencoder_gpt2-small_blocks.1.ln2.hook_normalized_24576.pt
INFO:src.lattmc.tc.transcoder_utils:Base GPT-2 and transcoder True layer=1 weights loaded successfully.
INFO:src.lattmc.tc.transcoder_utils:Transcoder for layer 1 loaded
INFO:src.lattmc.tc.transcoder_utils:Loading transcoder for layer 3
INFO:httpx:HTTP Request: HEAD https://huggingface.co/pchlenski/gpt2-transcoders/resolve/main/final_sparse_autoencoder_gpt2-small_blocks.3.ln2.hook_normalized_24576.pt "HTTP/1.1 302 Found"
INFO:src.lattmc.tc.transcoder_utils:Loaded file final_sparse_autoencoder_gpt2-small_blocks.3.ln2.hook_normalized_24576.pt
INFO:src.lattmc.tc.transcoder_utils:Base GPT-2 and transcoder True layer=3 w

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/generation_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:src.lattmc.tc.transcoder_utils:Initializing Tran

Loaded pretrained model gpt2-small into HookedTransformer
Moving model to device:  mps


INFO:httpx:HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:src.lattmc.tc.transcoder_utils:Transcoder initialized
INFO:src.lattmc.tc.transcoder_fca:Creating V for layer 1
INFO:src.lattmc.tc.transcoder_fca:/Users/ltsinadze/git/my_papers/notebooks/transcoders/data/transcoders/gpt2/V1.npz exists
INFO:src.lattmc.tc.transcod

In [25]:
empty_cache()

0

In [26]:
with tqdm(layers) as players:
    nz_fcas = {layer: NonzeroFCA.fromFCA(tr_analyzer.fcas[layer]) for layer in players}
    nz_analyzer = tr_analyzer.fromAnalyzer(
        tr_analyzer,
        fcas = nz_fcas,
    )

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:05<00:00,  1.96s/it]


In [27]:
empty_cache()

0

In [28]:
tr_analyzer.transcoder.model.cfg.device

'mps'

In [29]:
empty_cache()

0

## Features Dimension

In [30]:
tr_analyzer.fcas[layers[0]].V.shape, tr_analyzer.fcas[layers[1]].V.shape, tr_analyzer.fcas[layers[-1]].V.shape, tr_analyzer.tokens.shape

((25600, 24576), (25600, 24576), (25600, 24576), torch.Size([25600, 128]))

## Select Sample $A \in X$

In [31]:
# A_nyc = np.random.randint(0, high=tr_analyzer.corpus.shape[0], size=11)
# A_nyc

In [32]:
# A_nyc = np.array([
#     25576, 
#     19744,  
#     4488,  
#     8217, 
#     21048, 
#     11717, 
#     15774,  
#     1431, 
#     19151,
#     15355, 
#     24614
# ])

In [33]:
# A_nyc = np.array([17525, 11882,  1305,  2024, 19824,  7857,  5753, 20095, 22451,
#          841, 10946])

In [34]:
# A_nyc = np.array([11882, 1305, 7857, 20095, 22451, 841, 10946])

In [35]:
# A_nyc = np.array([1305, 7857, 22451, 841, 10946])

In [36]:
A_nyc = np.array([25314,  7705, 25413,  1757,  9276, 19655, 23050, 18379, 19094,
        8099, 17518, 13766, 20164,  2949, 15098, 11772, 23708])

## Mapping on layer 1

In [37]:
layer = 1

#### Calculate $G_{I}F_{I}$

In [38]:
conA_1 = tr_analyzer.fcas[layer].GF_F(A_nyc)
val, idcs = topNonZeros(conA_1.v)
val, idcs, val.shape

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25600/25600 [00:00<00:00, 33673.66it/s]


(array([ 2.3778465 , 11.623092  , 15.590901  , 17.185507  , 20.36225   ,
        18.157068  , 13.04484   , 14.283718  ,  1.3420291 ,  0.885348  ,
        10.77267   ,  0.8607995 , 17.016958  ,  0.7531933 , 17.257683  ,
        17.855234  ,  0.9846369 ,  0.1505084 ,  3.0616767 ,  0.16389146,
        17.51136   ,  9.489657  ,  0.5782434 , 17.580696  , 13.000805  ,
         0.05045986,  0.20622489,  0.2671957 ,  0.22487435, 21.153797  ,
         0.22075982,  9.437564  , 17.576523  ,  9.969621  ,  0.3783932 ,
         6.4114914 ,  3.3060553 ,  7.3076477 , 14.873096  ,  1.9929867 ,
        12.562787  , 12.955371  , 17.943571  , 16.983948  , 17.658993  ,
         5.0929074 ,  9.490306  ,  1.053812  , 20.465458  ,  0.55529314,
         0.3689103 , 11.243937  ,  0.6720834 ,  3.7129402 , 16.691938  ],
       dtype=float32),
 array([  272,   669,  1371,  1622,  1775,  2401,  2911,  2929,  2948,
         4814,  6011,  6317,  6859,  7534,  7590,  8632,  8964,  9474,
         9645,  9831,  9965, 10

In [39]:
A_nyc.shape, conA_1.A.shape

((17,), (4974,))

In [40]:
_, idxs = topNonZeros(conA_1.v)
max_idx = idxs.shape[0]
max_idx

55

In [41]:
p_k = topRangeProject(conA_1.v, from_idx=10, to_idx=25)
topK(p_k, k=5)

(array([17.257683, 17.185507, 17.016958, 16.983948, 16.691938],
       dtype=float32),
 array([ 7590,  1622,  6859, 18918, 24065]))

In [42]:
p_test = topRangeProject(conA_1.v, from_idx=0, to_idx=max_idx)
topK(p_test, k=7), p_test.shape

((array([21.153797, 20.465458, 20.36225 , 18.157068, 17.943571, 17.855234,
         17.658993], dtype=float32),
  array([14997, 21260,  1775,  2401, 18914,  8632, 19007])),
 (24576,))

In [43]:
A_sub = np.array([
    25314,
    7705,
    25413,
    1757,
    9276
])

In [44]:
A_nyc[:5]

array([25314,  7705, 25413,  1757,  9276])

In [45]:
save_tmp(conA_1.A)

In [47]:
_ = tr_analyzer.print_all_from_objects(
    layer,
    A_nyc[:5],
    conA_1.v,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

INFO:src.lattmc.tc.transcoder_utils:Prompt: index 25314: <|endoftext|>, Jack and editor Bruce Mohl won first place for In-Depth Reporting from the Association of Capitol Reporters and Editors for a look at special education funding in Massachusetts. The same organization also awarded first place to a unique collaboration between WFXT-TV (FOX25) and CommonWealth for a series of stories on the Boston Redevelopment Authority and city employees getting affordable housing units, written by Jack and Bruce. A Boston native, Jack has lived in Massachusetts all his life. He was a major in English and history with a minor in political science at the University of Massachusetts, Boston. A father and grandfather, he

INFO:src.lattmc.tc.transcoder_utils:Prompt: index 7705: <|endoftext|>, so Shanahan said the defensive backs will have to use their athleticism and leaping ability to try to combat Brady''s favorite red zone target.

WSH running back Roy Helu vs. NE middle linebacker Brandon Spikes

Th

In [49]:
multitxt_l0 = tr_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    conA_1.v,
    limit=5,
)

Assembling text: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 128/128 [00:00<00:00, 114495.82it/s]
INFO:src.lattmc.tc.transcoder_utils:Prompt 25314: <|endoftext|>, Jack and editor Bruce Mohl won first place for In-Depth Reporting from the Association of Capitol Reporters and Editors for a look at special education funding in Massachusetts. The same organization also awarded first place to a unique collaboration between WFXT-TV (FOX25) and CommonWealth for a series of stories on the Boston Redevelopment Authority and city employees getting affordable housing units, written by Jack and Bruce. A Boston native, Jack has lived in Massachusetts all his life. He was a major in English and history with a minor in political science at the University of Massachusetts, Boston. A father and grandfather, he
Assembling text: 100%|██████████████████████████████████████████████████████████████████████████████████████████

In [50]:
print_latex_text(multitxt_l0)

Printing texts: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 7946.77it/s]

\item{ID 25314:}
  "\ldots\colorbox[HTML]{D0F1D7}{}\colorbox[HTML]{BEEFDE}{,}
  \colorbox[HTML]{D7F1D0}{Jack} \colorbox[HTML]{D0D0F1}{and}
  \colorbox[HTML]{EDD0F1}{editor} \colorbox[HTML]{D7F1D0}{Bruce}
  \colorbox[HTML]{EDD0F1}{Moh}l \colorbox[HTML]{D0D0F1}{won}
  \colorbox[HTML]{D0D0F1}{first} \colorbox[HTML]{D0D0F1}{place}
  \colorbox[HTML]{D0D0F1}{for}
  \colorbox[HTML]{D0D0F1}{In}\colorbox[HTML]{D0D0F1}{-}\colorbox[HTML]{D0D7F1}{Depth}
  Reporting \colorbox[HTML]{D0D7F1}{from} \colorbox[HTML]{F1D0D7}{the}
  \colorbox[HTML]{EDD0F1}{Association} \colorbox[HTML]{F1DFD0}{of}
  \colorbox[HTML]{EDD0F1}{Capitol} Reporters
  \colorbox[HTML]{D0D7F1}{and} Editors \colorbox[HTML]{F1DFD0}{for}
  \colorbox[HTML]{BEE9EF}{a} look \colorbox[HTML]{D0D7F1}{at} special
  education funding \colorbox[HTML]{BEE9EF}{in}
  \colorbox[HTML]{EDD0F1}{Massachusetts}\colorbox[HTML]{D0D7F1}{.}
  \colorbox[HTML]{EFDEBE}{The} same organization also awarded first
  \colorbox[HTML]{BEE9EF}{place} \colorbox[HTML]{B

In [51]:
pres = tr_analyzer.print_all_from_objects(
    layer,
    conA_1.A,
    p_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

INFO:src.lattmc.tc.transcoder_utils:Prompt: index 1: <|endoftext|> political in nature.

"According to Sangh system, office bearers cannot work in the political sphere. Subhash Velingkar cannot accept both responsibilities, that of being a sangha chalak and participate in politics. Therefore, he has been relieved of responsibilities in order to allow him to dedicate himself for BBSM (Bharatiya Bhasha Suraksha Manch) work," Behare said, adding that while the agitation for education in the mother tongue in primary schools was justified, his original RSS faction would not participate in it.

"BBSM's work has support of the Sangh

INFO:src.lattmc.tc.transcoder_utils:Prompt: index 7: <|endoftext|>o, then 23, performed a rousing, soulful version of it on acoustic guitar. The stadium''s switchboard received over 2,000 calls, and NBC also received hundreds of angry calls. Feliciano said later it brought his career to a temporary halt, that for a while radio stations refused to play his records

In [52]:
_ = tr_analyzer.print_multitexts_from_objects(
    layer,
    # conA.A,
    A_nyc[:5],
    p_test,
    limit=5,
)

Assembling text: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 128/128 [00:00<00:00, 109677.41it/s]
INFO:src.lattmc.tc.transcoder_utils:Prompt 25314: <|endoftext|>, Jack and editor Bruce Mohl won first place for In-Depth Reporting from the Association of Capitol Reporters and Editors for a look at special education funding in Massachusetts. The same organization also awarded first place to a unique collaboration between WFXT-TV (FOX25) and CommonWealth for a series of stories on the Boston Redevelopment Authority and city employees getting affordable housing units, written by Jack and Bruce. A Boston native, Jack has lived in Massachusetts all his life. He was a major in English and history with a minor in political science at the University of Massachusetts, Boston. A father and grandfather, he
Assembling text: 100%|██████████████████████████████████████████████████████████████████████████████████████████

### Minimal FCA $G_{J}F_{J}$ and Analysis

In [ ]:
nozA = nz_analyzer.fcas[layer].GF_F(A_nyc)
val, idcs = topNonZeros(nozA.v)
val, idcs, val.shape

In [ ]:
A_nyc.shape, nozA.A.shape

In [ ]:
n_k = topRangeProject(nozA.v, from_idx=10, to_idx=25)
topK(n_k, k=5)

In [ ]:
n_test = topRangeProject(nozA.v, from_idx=0, to_idx=max_idx)
topK(n_test, k=5), n_test.shape

In [ ]:
pres = nz_analyzer.print_all_from_objects(
    layer,
    A_nyc,
    n_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
multitxt_z0 = nz_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    n_test,
    limit=5,
)

In [ ]:
print_latex_text(multitxt_z0)

In [ ]:
pres = nz_analyzer.print_all_from_objects(
    layer,
    nozA.A,
    n_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
_ = nz_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    n_test,
    limit=5,
)

In [ ]:
intA = intersect(conA.A, nozA.A)
np.all(intA == conA.A)

## Mapping on layer 3

In [ ]:
layer = 3

#### Calculate $G_{I}F_{I}$

In [53]:
conA_3 = tr_analyzer.fcas[layer].GF_F(A_nyc)
val, idcs = topNonZeros(conA_3.v)
val, idcs, val.shape

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25600/25600 [00:00<00:00, 27339.07it/s]


(array([ 2.3778465 , 11.623092  , 15.590901  , 17.185507  , 20.36225   ,
        18.157068  , 13.04484   , 14.283718  ,  1.3420291 ,  0.885348  ,
        10.77267   ,  0.8607995 , 17.016958  ,  0.7531933 , 17.257683  ,
        17.855234  ,  0.9846369 ,  0.1505084 ,  3.0616767 ,  0.16389146,
        17.51136   ,  9.489657  ,  0.5782434 , 17.580696  , 13.000805  ,
         0.05045986,  0.20622489,  0.2671957 ,  0.22487435, 21.153797  ,
         0.22075982,  9.437564  , 17.576523  ,  9.969621  ,  0.3783932 ,
         6.4114914 ,  3.3060553 ,  7.3076477 , 14.873096  ,  1.9929867 ,
        12.562787  , 12.955371  , 17.943571  , 16.983948  , 17.658993  ,
         5.0929074 ,  9.490306  ,  1.053812  , 20.465458  ,  0.55529314,
         0.3689103 , 11.243937  ,  0.6720834 ,  3.7129402 , 16.691938  ],
       dtype=float32),
 array([  272,   669,  1371,  1622,  1775,  2401,  2911,  2929,  2948,
         4814,  6011,  6317,  6859,  7534,  7590,  8632,  8964,  9474,
         9645,  9831,  9965, 10

In [54]:
A_nyc.shape, conA_3.A.shape

((17,), (4974,))

In [55]:
_, idxs = topNonZeros(conA_3.v)
max_idx = idxs.shape[0]
max_idx

55

In [56]:
p_k = topRangeProject(conA_3.v, from_idx=10, to_idx=25)
topK(p_k, k=5)

(array([17.257683, 17.185507, 17.016958, 16.983948, 16.691938],
       dtype=float32),
 array([ 7590,  1622,  6859, 18918, 24065]))

In [57]:
p_test = topRangeProject(conA_3.v, from_idx=0, to_idx=max_idx)
topK(p_test, k=7), p_test.shape

((array([21.153797, 20.465458, 20.36225 , 18.157068, 17.943571, 17.855234,
         17.658993], dtype=float32),
  array([14997, 21260,  1775,  2401, 18914,  8632, 19007])),
 (24576,))

In [58]:
A_sub = np.array([
    25314,
    7705,
    25413,
    1757,
    9276
])

In [59]:
A_nyc[:5]

array([25314,  7705, 25413,  1757,  9276])

In [60]:
_ = tr_analyzer.print_all_from_objects(
    layer,
    A_nyc[:5],
    conA_3.v,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

INFO:src.lattmc.tc.transcoder_utils:Prompt: index 25314: <|endoftext|>, Jack and editor Bruce Mohl won first place for In-Depth Reporting from the Association of Capitol Reporters and Editors for a look at special education funding in Massachusetts. The same organization also awarded first place to a unique collaboration between WFXT-TV (FOX25) and CommonWealth for a series of stories on the Boston Redevelopment Authority and city employees getting affordable housing units, written by Jack and Bruce. A Boston native, Jack has lived in Massachusetts all his life. He was a major in English and history with a minor in political science at the University of Massachusetts, Boston. A father and grandfather, he

INFO:src.lattmc.tc.transcoder_utils:Prompt: index 7705: <|endoftext|>, so Shanahan said the defensive backs will have to use their athleticism and leaping ability to try to combat Brady''s favorite red zone target.

WSH running back Roy Helu vs. NE middle linebacker Brandon Spikes

Th

In [61]:
multitxt_l0 = tr_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    conA_3.v,
    limit=5,
)

Assembling text: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 128/128 [00:00<00:00, 119623.64it/s]
INFO:src.lattmc.tc.transcoder_utils:Prompt 25314: <|endoftext|>, Jack and editor Bruce Mohl won first place for In-Depth Reporting from the Association of Capitol Reporters and Editors for a look at special education funding in Massachusetts. The same organization also awarded first place to a unique collaboration between WFXT-TV (FOX25) and CommonWealth for a series of stories on the Boston Redevelopment Authority and city employees getting affordable housing units, written by Jack and Bruce. A Boston native, Jack has lived in Massachusetts all his life. He was a major in English and history with a minor in political science at the University of Massachusetts, Boston. A father and grandfather, he
Assembling text: 100%|██████████████████████████████████████████████████████████████████████████████████████████

In [62]:
print_latex_text(multitxt_l0)

Printing texts: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 10087.31it/s]

\item{ID 25314:}
  "\ldots\colorbox[HTML]{D0F1D7}{}\colorbox[HTML]{BEEFDE}{,}
  \colorbox[HTML]{D7F1D0}{Jack} \colorbox[HTML]{D0D0F1}{and}
  \colorbox[HTML]{EDD0F1}{editor} \colorbox[HTML]{D7F1D0}{Bruce}
  \colorbox[HTML]{EDD0F1}{Moh}l \colorbox[HTML]{D0D0F1}{won}
  \colorbox[HTML]{D0D0F1}{first} \colorbox[HTML]{D0D0F1}{place}
  \colorbox[HTML]{D0D0F1}{for}
  \colorbox[HTML]{D0D0F1}{In}\colorbox[HTML]{D0D0F1}{-}\colorbox[HTML]{D0D7F1}{Depth}
  Reporting \colorbox[HTML]{D0D7F1}{from} \colorbox[HTML]{F1D0D7}{the}
  \colorbox[HTML]{EDD0F1}{Association} \colorbox[HTML]{F1DFD0}{of}
  \colorbox[HTML]{EDD0F1}{Capitol} Reporters
  \colorbox[HTML]{D0D7F1}{and} Editors \colorbox[HTML]{F1DFD0}{for}
  \colorbox[HTML]{BEE9EF}{a} look \colorbox[HTML]{D0D7F1}{at} special
  education funding \colorbox[HTML]{BEE9EF}{in}
  \colorbox[HTML]{EDD0F1}{Massachusetts}\colorbox[HTML]{D0D7F1}{.}
  \colorbox[HTML]{EFDEBE}{The} same organization also awarded first
  \colorbox[HTML]{BEE9EF}{place} \colorbox[HTML]{B

In [63]:
pres = tr_analyzer.print_all_from_objects(
    layer,
    conA_3.A,
    p_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

INFO:src.lattmc.tc.transcoder_utils:Prompt: index 1: <|endoftext|> political in nature.

"According to Sangh system, office bearers cannot work in the political sphere. Subhash Velingkar cannot accept both responsibilities, that of being a sangha chalak and participate in politics. Therefore, he has been relieved of responsibilities in order to allow him to dedicate himself for BBSM (Bharatiya Bhasha Suraksha Manch) work," Behare said, adding that while the agitation for education in the mother tongue in primary schools was justified, his original RSS faction would not participate in it.

"BBSM's work has support of the Sangh

INFO:src.lattmc.tc.transcoder_utils:Prompt: index 7: <|endoftext|>o, then 23, performed a rousing, soulful version of it on acoustic guitar. The stadium''s switchboard received over 2,000 calls, and NBC also received hundreds of angry calls. Feliciano said later it brought his career to a temporary halt, that for a while radio stations refused to play his records

In [64]:
_ = tr_analyzer.print_multitexts_from_objects(
    layer,
    # conA.A,
    A_nyc[:5],
    p_test,
    limit=5,
)

Assembling text: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 128/128 [00:00<00:00, 120753.69it/s]
INFO:src.lattmc.tc.transcoder_utils:Prompt 25314: <|endoftext|>, Jack and editor Bruce Mohl won first place for In-Depth Reporting from the Association of Capitol Reporters and Editors for a look at special education funding in Massachusetts. The same organization also awarded first place to a unique collaboration between WFXT-TV (FOX25) and CommonWealth for a series of stories on the Boston Redevelopment Authority and city employees getting affordable housing units, written by Jack and Bruce. A Boston native, Jack has lived in Massachusetts all his life. He was a major in English and history with a minor in political science at the University of Massachusetts, Boston. A father and grandfather, he
Assembling text: 100%|██████████████████████████████████████████████████████████████████████████████████████████

### Minimal FCA $G_{J}F_{J}$ and Analysis

In [ ]:
nozA = nz_analyzer.fcas[layer].GF_F(A_nyc)
val, idcs = topNonZeros(nozA.v)
val, idcs, val.shape

In [ ]:
A_nyc.shape, nozA.A.shape

In [ ]:
n_k = topRangeProject(nozA.v, from_idx=10, to_idx=25)
topK(n_k, k=5)

In [ ]:
n_test = topRangeProject(nozA.v, from_idx=0, to_idx=max_idx)
topK(n_test, k=5), n_test.shape

In [ ]:
pres = nz_analyzer.print_all_from_objects(
    layer,
    A_nyc,
    n_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
multitxt_z0 = nz_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    n_test,
    limit=5,
)

In [ ]:
print_latex_text(multitxt_z0)

In [ ]:
pres = nz_analyzer.print_all_from_objects(
    layer,
    nozA.A,
    n_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
_ = nz_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    n_test,
    limit=5,
)

In [ ]:
intA = intersect(conA.A, nozA.A)
np.all(intA == conA.A)

## Mapping on layer 9

In [ ]:
layer = 9

#### Calculate $G_{I}F_{I}$

In [65]:
conA_9 = tr_analyzer.fcas[layer].GF_F(A_nyc)
val, idcs = topNonZeros(conA_9.v)
val, idcs, val.shape

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25600/25600 [00:00<00:00, 35561.48it/s]


(array([ 2.3778465 , 11.623092  , 15.590901  , 17.185507  , 20.36225   ,
        18.157068  , 13.04484   , 14.283718  ,  1.3420291 ,  0.885348  ,
        10.77267   ,  0.8607995 , 17.016958  ,  0.7531933 , 17.257683  ,
        17.855234  ,  0.9846369 ,  0.1505084 ,  3.0616767 ,  0.16389146,
        17.51136   ,  9.489657  ,  0.5782434 , 17.580696  , 13.000805  ,
         0.05045986,  0.20622489,  0.2671957 ,  0.22487435, 21.153797  ,
         0.22075982,  9.437564  , 17.576523  ,  9.969621  ,  0.3783932 ,
         6.4114914 ,  3.3060553 ,  7.3076477 , 14.873096  ,  1.9929867 ,
        12.562787  , 12.955371  , 17.943571  , 16.983948  , 17.658993  ,
         5.0929074 ,  9.490306  ,  1.053812  , 20.465458  ,  0.55529314,
         0.3689103 , 11.243937  ,  0.6720834 ,  3.7129402 , 16.691938  ],
       dtype=float32),
 array([  272,   669,  1371,  1622,  1775,  2401,  2911,  2929,  2948,
         4814,  6011,  6317,  6859,  7534,  7590,  8632,  8964,  9474,
         9645,  9831,  9965, 10

In [66]:
A_nyc.shape, conA_9.A.shape

((17,), (4974,))

In [67]:
_, idxs = topNonZeros(conA_9.v)
max_idx = idxs.shape[0]
max_idx

55

In [68]:
conA_9.A.shape, conA_3.A.shape

((4974,), (4974,))

In [84]:
conA_3_GF = tr_analyzer.fcas[3].GF_F(conA_9.A)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25600/25600 [00:00<00:00, 34997.82it/s]


In [85]:
conA_9.A.shape, conA_3_GF.A.shape

((4974,), (25600,))

In [86]:
topNonZeros(conA_9.v)

(array([ 2.3778465 , 11.623092  , 15.590901  , 17.185507  , 20.36225   ,
        18.157068  , 13.04484   , 14.283718  ,  1.3420291 ,  0.885348  ,
        10.77267   ,  0.8607995 , 17.016958  ,  0.7531933 , 17.257683  ,
        17.855234  ,  0.9846369 ,  0.1505084 ,  3.0616767 ,  0.16389146,
        17.51136   ,  9.489657  ,  0.5782434 , 17.580696  , 13.000805  ,
         0.05045986,  0.20622489,  0.2671957 ,  0.22487435, 21.153797  ,
         0.22075982,  9.437564  , 17.576523  ,  9.969621  ,  0.3783932 ,
         6.4114914 ,  3.3060553 ,  7.3076477 , 14.873096  ,  1.9929867 ,
        12.562787  , 12.955371  , 17.943571  , 16.983948  , 17.658993  ,
         5.0929074 ,  9.490306  ,  1.053812  , 20.465458  ,  0.55529314,
         0.3689103 , 11.243937  ,  0.6720834 ,  3.7129402 , 16.691938  ],
       dtype=float32),
 array([  272,   669,  1371,  1622,  1775,  2401,  2911,  2929,  2948,
         4814,  6011,  6317,  6859,  7534,  7590,  8632,  8964,  9474,
         9645,  9831,  9965, 10

In [87]:
topNonZeros(conA_3_GF.v)

(array([18.548979 , 21.262352 , 19.127148 ,  6.0022683, 16.03913  ,
        14.319236 , 17.895407 , 12.876327 , 15.551559 ,  8.680254 ,
        18.81791  , 14.998608 , 14.472126 ], dtype=float32),
 array([ 5645,  8175,  8262,  8893, 12337, 14301, 14326, 15354, 16634,
        17241, 20707, 23592, 24077]))

In [79]:
B_GF_diff = diff(conA_3_GF.A, conA_9_GF.A)
B_GF_diff.shape

(20626,)

In [ ]:
B_diff = diff(intersect(conA_3.A, conA_9.A), A_nyc)
B_diff.shape

In [ ]:
intersect(A_nyc, B_diff)

In [ ]:
A_diff = diff(conA_3.A, conA_9.A)
A_diff.shape

In [ ]:
con_9_diff = tr_analyzer.fcas[9].GF_F(B_diff)

In [ ]:
con_9_diff.A

In [ ]:
subset(A_nyc, con_9_diff.A)

In [ ]:
v = meet(conA_9.v, con_9_diff.v)
topNonZeros(v)

In [ ]:
topNonZeros(con_9_diff.v)

In [ ]:
G_v = tr_analyzer.fcas[layer].G(v)
subset(B_diff, G_v), G_v.shape

In [ ]:
le(conA_9.v, con_9_diff.v), le(con_9_diff.v, conA_9.v)

In [ ]:
p_k = topRangeProject(conA_9.v, from_idx=10, to_idx=25)
topK(p_k, k=5)

In [ ]:
p_test = topRangeProject(v, from_idx=0, to_idx=25)
topK(p_test, k=7), p_test.shape

In [ ]:
le(p_test, v)

In [ ]:
pres = tr_analyzer.print_all_from_objects(
    layer,
    B_diff,
    v,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
multitxt_l8 = tr_analyzer.print_multitexts_from_objects(
    layer,
    B_diff[:5],
    v,
    limit=5,
)

In [ ]:
print_latex_text(multitxt_l8)

In [ ]:
pres = tr_analyzer.print_all_from_objects(
    layer,
    conA.A,
    p_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
layer = 9
dims = np.flatnonzero(p_test)

for prompt_idx in B_diff[:5]:
    vs = tr_analyzer.transcoder(tr_analyzer.tokens[prompt_idx], layer)[0]
    mx = vs[:, dims].max(axis=0)
    pos = vs[:, dims].argmax(axis=0)
    print(prompt_idx, list(zip(dims, p_test[dims], mx, pos, mx >= p_test[dims])))


In [ ]:
vs = tr_analyzer.transcoder(tr_analyzer.tokens[prompt_idx], layer)[0]

In [ ]:
mx = vs[:, dims].max(axis=0)
mx

In [ ]:
pos = vs[:, dims].argmax(axis=0)
pos

In [ ]:
dims

In [ ]:
p_5463 = np.zeros_like(p_test)
p_5463[5463] = p_test[5463]
p_pr = project_on(p_test, 22194)
le(p_5463, v), le(p_pr, v)

In [ ]:
mask = upper_mask(p_pr, vs)
mask

In [ ]:
upper_indices(p_pr, vs)

In [ ]:
np.where(mask)[0]

In [ ]:
v_all = np.array([np.all(p_pr <= v_i) for v_i in vs])
v_all.shape

In [ ]:
v_vec = np.all(p_pr <= vs, axis=1)
v_vec.shape

In [ ]:
np.all(v_all == v_vec, axis = 0)

In [ ]:
np.all(p_pr <= vs[0], axis=0)

In [ ]:
_ = tr_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    p_test,
    limit=5,
)

### Minimal FCA $G_{J}F_{J}$ and Analysis

In [ ]:
nozA = nz_analyzer.fcas[layer].GF_F(A_nyc)
val, idcs = topNonZeros(nozA.v)
val, idcs, val.shape

In [ ]:
A_nyc.shape, nozA.A.shape

In [ ]:
n_k = topRangeProject(nozA.v, from_idx=10, to_idx=25)
topK(n_k, k=5)

In [ ]:
n_test = topRangeProject(nozA.v, from_idx=0, to_idx=max_idx)
topK(n_test, k=5), n_test.shape

In [ ]:
pres = nz_analyzer.print_all_from_objects(
    layer,
    A_nyc,
    n_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
multitxt_z8 = nz_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    n_test,
    limit=5,
)

In [ ]:
print_latex_text(multitxt_z8)

In [ ]:
pres = nz_analyzer.print_all_from_objects(
    layer,
    nozA.A,
    n_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
_ = nz_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    n_test,
    limit=5,
)

In [ ]:
intA = intersect(conA.A, nozA.A)
np.all(intA == conA.A)

## Experiments on Token FCA

In [ ]:
tr_analyzer.tokenize('New').to('cpu')[0][-1]

In [ ]:
T = [
    tr_analyzer.tokenize(' New').to('cpu').tolist()[0][-1],
    tr_analyzer.tokenize(' York').to('cpu').tolist()[0][-1],
    tr_analyzer.tokenize(' City').to('cpu').tolist()[0][-1],
    # tr_analyzer.tokenize(' Official').to('cpu').tolist()[0][-1],
]
T

In [ ]:
S_ny = {968, 1971}
S_ny

In [ ]:
tr_analyzer.transcoder.print_tokens_with_text(T)

In [ ]:
T

### Initialize FCA for tokens

In [ ]:
pca = TokenFCA(
    tr_analyzer.corpus,
    T
)

In [ ]:
S = set(T)
S, T[0], T[1], is_empty(S)

In [ ]:
G_S = pca.G(S)
G_S

In [ ]:
F_S = pca.F(G_S)
F_S, S

In [ ]:
powS = powerset_bit(S)
powS, pca.powS

In [ ]:
cl_arr = pca.are_attributes_closed()
cl_arr

In [ ]:
G_S1 = pca.G({T[0]})
G_S2 = pca.G({T[1]})
G_S12 = intersect(G_S1, G_S2)
G_S = pca.G(S)

In [ ]:
G_S1.shape, G_S2.shape, G_S12.shape, G_S.shape, np.array_equal(G_S, G_S12) 

In [ ]:
_ = tr_analyzer.match_tokens_in_prompts_and_print(G_S[:5], S)

In [ ]:
G_S_ny = pca.G(S_ny)

In [ ]:
S_ny

In [ ]:
G_S_diff = diff(G_S_ny, G_S)

In [ ]:
text_dict = tr_analyzer.match_tokens_in_prompts_and_print(G_S_diff[:5], S_ny, color_idx=35)

In [ ]:
print(print_latex_text(text_dict))

In [ ]:
S_ny = set(T[:2])
S_ny, tr_analyzer.to_clean(T[:2])

In [ ]:
G_Ts = list()
for t in S_ny:
    G_t = pca.G(set({t}))
    G_Ts.append(G_t)

## Init Infomorphisms

In [ ]:
infoms = {layer: FCAtoPCA(
    tr_analyzer.fcas[layer],
    pca
) for layer in layers}

### Map on Layer 0

In [ ]:
layer = 0

In [ ]:
infom = infoms[layer]

In [ ]:
infoms

In [ ]:
infom.g_st

In [ ]:
infom.powS

In [ ]:
G_S_ny = pca.G(S_ny)
G_S_ny.shape

In [ ]:
infom.g(S_ny)

In [ ]:
A_ny = G_S_ny[:9]
A_ny

In [ ]:
infom.powS[[0, 2]]

In [ ]:
f_A = infom.f(A_ny)
f_A.shape

In [ ]:
subset(A_ny, f_A)

In [ ]:
intersect(A_ny, f_A)

In [ ]:
G_I_F_I_A_ny = infom.fca.GF(A_ny)
f_G_I_F_I_A_ny = infom.f(G_I_F_I_A_ny)
f_A_ny = infom.f(A_ny)
G_M_F_M_f_A_ny = infom.pca.GF(f_A_ny)

In [ ]:
f_G_I_F_I_A_ny.shape, G_M_F_M_f_A_ny.shape

In [ ]:
subset(f_G_I_F_I_A_ny, G_M_F_M_f_A_ny)

In [ ]:
gS = infom.g(S_ny)
topNonZeros(gS)

In [ ]:
infom.fca.G(gS)

In [ ]:
pre_S_f = infom.pre_f(A_ny)
pre_S_f

In [ ]:
f_A

In [ ]:
text_dict = tr_analyzer.match_tokens_in_prompts_and_print(f_A[:5], pre_S_f, color_idx=35)

In [ ]:
print_latex_text(text_dict)

In [ ]:
f_A_sam = [key.item() for key in text_dict.keys()]
f_A_sam

In [ ]:
f_A_stat = [11136, 3457, 22529, 20356, 11014]

### Map on Layer 8

In [ ]:
layer = 7

In [ ]:
infom = infoms[layer]

In [ ]:
infoms

In [ ]:
infom.g_st

In [ ]:
infom.powS

In [ ]:
G_S_ny = pca.G(S_ny)
G_S_ny.shape

In [ ]:
infom.g(S_ny)

In [ ]:
A_ny = G_S_ny[:9]
A_ny

In [ ]:
infom.powS[[0, 2]]

In [ ]:
f_A = infom.f(A_ny)
f_A.shape

In [ ]:
subset(A_ny, f_A)

In [ ]:
intersect(A_ny, f_A)

In [ ]:
G_I_F_I_A_ny = infom.fca.GF(A_ny)
f_G_I_F_I_A_ny = infom.f(G_I_F_I_A_ny)
f_A_ny = infom.f(A_ny)
G_M_F_M_f_A_ny = infom.pca.GF(f_A_ny)

In [ ]:
f_G_I_F_I_A_ny.shape, G_M_F_M_f_A_ny.shape

In [ ]:
subset(f_G_I_F_I_A_ny, G_M_F_M_f_A_ny)

In [ ]:
gS = infom.g(S_ny)
topNonZeros(gS)

In [ ]:
infom.fca.G(gS)

In [ ]:
pre_S_f = infom.pre_f(A_ny)
pre_S_f

In [ ]:
f_A

In [ ]:
text_dict = tr_analyzer.match_tokens_in_prompts_and_print(f_A[:5], pre_S_f, color_idx=35)

In [ ]:
print_latex_text(text_dict)